# 01_02 - Data Cleaning

This notebook cleans each raw dataset on its own, one at a time - calendar first, then weather, then consumption. Every column-keep/drop and missing-value decision is made and reviewed before moving to the next dataset. Joining the cleaned datasets together comes only after all three are done.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

import pandas as pd
from common.data_loading import load_calendar, load_weather, load_consumption
from common.cleaning import clean_calendar, clean_weather, clean_consumption
from common.joining import build_fsa_hourly_dataset, build_all_fsa_dataset

## 1. Calendar

Starting with the raw calendar file, before deciding anything about it.

In [2]:
calendar_raw = load_calendar()

print("shape:", calendar_raw.shape)
calendar_raw.info()

shape: (52584, 45)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52584 entries, 0 to 52583
Data columns (total 45 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   timestamp_utc            52584 non-null  object 
 1   timestamp_local          52584 non-null  object 
 2   timestamp_toronto        52584 non-null  object 
 3   date                     52584 non-null  object 
 4   year                     52584 non-null  int64  
 5   quarter                  52584 non-null  int64  
 6   month                    52584 non-null  int64  
 7   month_name               52584 non-null  object 
 8   week_of_year             52584 non-null  int64  
 9   day_of_year              52584 non-null  int64  
 10  day_of_month             52584 non-null  int64  
 11  hour                     52584 non-null  int64  
 12  hour_group               52584 non-null  object 
 13  weekday                  52584 non-null  int64  
 14  wee

In [3]:
calendar_raw.head()

,timestamp_utc,timestamp_local,timestamp_toronto,date,year,quarter,month,month_name,week_of_year,day_of_year,...,is_spring_forward_day,is_fall_back_day,utc_offset_hours,date_index,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos
0,2021-01-01 05:00:00+00:00,2021-01-01 00:00:00,2021-01-01 00:00:00-05:00,2021-01-01,2021,1,1,January,53,1,...,0,0,-5,0,0.000000,1.000000,-0.433884,-0.900969,0.0,1.0
1,2021-01-01 06:00:00+00:00,2021-01-01 01:00:00,2021-01-01 01:00:00-05:00,2021-01-01,2021,1,1,January,53,1,...,0,0,-5,0,0.258819,0.965926,-0.433884,-0.900969,0.0,1.0
2,2021-01-01 07:00:00+00:00,2021-01-01 02:00:00,2021-01-01 02:00:00-05:00,2021-01-01,2021,1,1,January,53,1,...,0,0,-5,0,0.500000,0.866025,-0.433884,-0.900969,0.0,1.0
3,2021-01-01 08:00:00+00:00,2021-01-01 03:00:00,2021-01-01 03:00:00-05:00,2021-01-01,2021,1,1,January,53,1,...,0,0,-5,0,0.707107,0.707107,-0.433884,-0.900969,0.0,1.0
4,2021-01-01 09:00:00+00:00,2021-01-01 04:00:00,2021-01-01 04:00:00-05:00,2021-01-01,2021,1,1,January,53,1,...,0,0,-5,0,0.866025,0.500000,-0.433884,-0.900969,0.0,1.0


Nine columns come out of the calendar file: `timestamp_utc`, `timestamp_toronto`, `date`, `month_name`, `weekday_name`, `hour_group`, `holiday_name`, `date_index` and `utc_offset_hours`. Each of these duplicates information that's already captured by another column we're keeping (raw numeric versions of the same fields, or a value that's a fixed function of `is_daylight_saving_time`) - it's about internal redundancy within this file, not a judgment on whether they matter for consumption.

Everything else stays as-is for now, including the DST flags and the cyclical encodings (`hour_sin`/`cos`, `weekday_sin`/`cos`, `month_sin`/`cos`): whether those actually help predict consumption can only be checked once calendar is joined with consumption, so pruning them here on intuition alone would risk throwing away something a correlation check might later show does matter.

The one other fix needed is `timestamp_local`, which loads as plain text (`object`) - it needs to be a real `datetime64` before it can be used as a join key or for any time-based operations.

In [4]:
calendar_clean = clean_calendar(calendar_raw)

print("shape:", calendar_clean.shape)
calendar_clean.info()

shape: (52584, 36)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52584 entries, 0 to 52583
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   timestamp_local          52584 non-null  datetime64[ns]
 1   year                     52584 non-null  int64         
 2   quarter                  52584 non-null  int64         
 3   month                    52584 non-null  int64         
 4   week_of_year             52584 non-null  int64         
 5   day_of_year              52584 non-null  int64         
 6   day_of_month             52584 non-null  int64         
 7   hour                     52584 non-null  int64         
 8   weekday                  52584 non-null  int64         
 9   is_weekend               52584 non-null  int64         
 10  is_workday               52584 non-null  int64         
 11  is_monday                52584 non-null  int64         
 12  is_friday    

In [5]:
calendar_clean.head()

,timestamp_local,year,quarter,month,week_of_year,day_of_year,day_of_month,hour,weekday,is_weekend,...,is_daylight_saving_time,is_dst_transition_day,is_spring_forward_day,is_fall_back_day,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos
0,2021-01-01 00:00:00,2021,1,1,53,1,1,0,4,0,...,0,0,0,0,0.000000,1.000000,-0.433884,-0.900969,0.0,1.0
1,2021-01-01 01:00:00,2021,1,1,53,1,1,1,4,0,...,0,0,0,0,0.258819,0.965926,-0.433884,-0.900969,0.0,1.0
2,2021-01-01 02:00:00,2021,1,1,53,1,1,2,4,0,...,0,0,0,0,0.500000,0.866025,-0.433884,-0.900969,0.0,1.0
3,2021-01-01 03:00:00,2021,1,1,53,1,1,3,4,0,...,0,0,0,0,0.707107,0.707107,-0.433884,-0.900969,0.0,1.0
4,2021-01-01 04:00:00,2021,1,1,53,1,1,4,4,0,...,0,0,0,0,0.866025,0.500000,-0.433884,-0.900969,0.0,1.0


In [6]:
# Confirm the two things this cleaning step was supposed to fix: no leftover nulls, and timestamp_local is a real datetime
print("timestamp_local dtype:", calendar_clean["timestamp_local"].dtype)
print("total nulls remaining:", calendar_clean.isna().sum().sum())

timestamp_local dtype: datetime64[ns]
total nulls remaining: 0


## 2. Weather

Two station files this time - Toronto City and Toronto INTL A (Pearson) - loaded separately before deciding anything about them, same as calendar.

In [7]:
weather_city_raw = load_weather("city")

print("shape:", weather_city_raw.shape)
weather_city_raw.info()

shape: (52584, 33)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52584 entries, 0 to 52583
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Longitude (x)        52584 non-null  float64
 1   Latitude (y)         52584 non-null  float64
 2   Station Name         52584 non-null  object 
 3   Climate ID           52584 non-null  int64  
 4   timestamp_local      52584 non-null  object 
 5   Year                 52584 non-null  int64  
 6   Month                52584 non-null  int64  
 7   Day                  52584 non-null  int64  
 8   Time (LST)           52584 non-null  object 
 9   Flag                 0 non-null      float64
 10  Temp (°C)            48794 non-null  float64
 11  Temp Flag            0 non-null      float64
 12  Dew Point Temp (°C)  48794 non-null  float64
 13  Dew Point Temp Flag  0 non-null      float64
 14  Rel Hum (%)          48794 non-null  float64
 15  Rel Hum Flag         0 non-null     

In [8]:
weather_city_raw.head()

,Longitude (x),Latitude (y),Station Name,Climate ID,timestamp_local,Year,Month,Day,Time (LST),Flag,...,Visibility Flag,Stn Press (kPa),Stn Press Flag,Hmdx,Hmdx Flag,Wind Chill,Wind Chill Flag,Weather,SOURCE_FILE,SOURCE_PERIOD
0,-79.4,43.67,TORONTO CITY,6158355,2021-01-01 00:00:00,2021,1,1,00:00,NaN,...,NaN,101.67,NaN,NaN,NaN,NaN,NaN,NaN,2021_01.csv,202101
1,-79.4,43.67,TORONTO CITY,6158355,2021-01-01 01:00:00,2021,1,1,01:00,NaN,...,NaN,101.66,NaN,NaN,NaN,NaN,NaN,NaN,2021_01.csv,202101
2,-79.4,43.67,TORONTO CITY,6158355,2021-01-01 02:00:00,2021,1,1,02:00,NaN,...,NaN,101.69,NaN,NaN,NaN,NaN,NaN,NaN,2021_01.csv,202101
3,-79.4,43.67,TORONTO CITY,6158355,2021-01-01 03:00:00,2021,1,1,03:00,NaN,...,NaN,101.77,NaN,NaN,NaN,NaN,NaN,NaN,2021_01.csv,202101
4,-79.4,43.67,TORONTO CITY,6158355,2021-01-01 04:00:00,2021,1,1,04:00,NaN,...,NaN,101.75,NaN,NaN,NaN,NaN,NaN,NaN,2021_01.csv,202101


In [9]:
weather_intl_raw = load_weather("intl_a")

print("shape:", weather_intl_raw.shape)
weather_intl_raw.info()

shape:

 (52584, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52584 entries, 0 to 52583
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Longitude (x)        52584 non-null  float64
 1   Latitude (y)         52584 non-null  float64
 2   Station Name         52584 non-null  object 
 3   Climate ID           52584 non-null  int64  
 4   timestamp_local      52584 non-null  object 
 5   Year                 52584 non-null  int64  
 6   Month                52584 non-null  int64  
 7   Day                  52584 non-null  int64  
 8   Time (LST)           52584 non-null  object 
 9   Flag                 0 non-null      float64
 10  Temp (°C)            48786 non-null  float64
 11  Temp Flag            0 non-null      float64
 12  Dew Point Temp (°C)  48785 non-null  float64
 13  Dew Point Temp Flag  1 non-null      object 
 14  Rel Hum (%)          48785 non-null  float64
 15  Rel Hum Flag         1 

In [10]:
weather_intl_raw.head()

,Longitude (x),Latitude (y),Station Name,Climate ID,timestamp_local,Year,Month,Day,Time (LST),Flag,...,Visibility Flag,Stn Press (kPa),Stn Press Flag,Hmdx,Hmdx Flag,Wind Chill,Wind Chill Flag,Weather,SOURCE_FILE,SOURCE_PERIOD
0,-79.63,43.68,TORONTO INTL A,6158731,2021-01-01 00:00:00,2021,1,1,00:00,NaN,...,NaN,100.93,NaN,NaN,NaN,-4.0,NaN,NaN,2021_01.csv,202101
1,-79.63,43.68,TORONTO INTL A,6158731,2021-01-01 01:00:00,2021,1,1,01:00,NaN,...,NaN,100.93,NaN,NaN,NaN,-3.0,NaN,Clear,2021_01.csv,202101
2,-79.63,43.68,TORONTO INTL A,6158731,2021-01-01 02:00:00,2021,1,1,02:00,NaN,...,NaN,100.95,NaN,NaN,NaN,-4.0,NaN,NaN,2021_01.csv,202101
3,-79.63,43.68,TORONTO INTL A,6158731,2021-01-01 03:00:00,2021,1,1,03:00,NaN,...,NaN,101.03,NaN,NaN,NaN,-5.0,NaN,NaN,2021_01.csv,202101
4,-79.63,43.68,TORONTO INTL A,6158731,2021-01-01 04:00:00,2021,1,1,04:00,NaN,...,NaN,101.02,NaN,NaN,NaN,-5.0,NaN,Mostly Cloudy,2021_01.csv,202101


`timestamp_local` loads as plain text here too, same issue as calendar - fixing that first, before deciding anything else about these two datasets.

Both stations share the exact same 33 columns in the same order, so the same cleaning logic applies to both - only how populated each column is differs between them.

Columns coming out fall into three groups: station metadata that's constant within a file (`Longitude (x)`, `Latitude (y)`, `Station Name`, `Climate ID`), fields duplicating what calendar or `timestamp_local` already cover (`Time (LST)`, `Year`, `Month`, `Day`, `SOURCE_FILE`, `SOURCE_PERIOD`), and columns that carry essentially no information - every `*Flag` column is 0% populated, and `Wind Chill`/`Hmdx`/`Weather` are all derived from (or a restatement of) the numeric measurements being kept, on top of being mostly null even within the station that reports them.

What stays: `Temp (°C)`, `Dew Point Temp (°C)`, `Rel Hum (%)`, `Precip. Amount (mm)`, `Wind Dir (10s deg)`, `Wind Spd (km/h)`, `Visibility (km)`, `Stn Press (kPa)`, plus `timestamp_local`. Precip/Wind/Visibility are the ones that differ sharply by station (~93% populated at the station that measures them, 0% at the other) - that's expected given each station only measures what it measures, not a reason to drop them.

In [11]:
weather_city_clean = clean_weather(weather_city_raw)
weather_intl_clean = clean_weather(weather_intl_raw)

print("weather_city shape:", weather_city_clean.shape)
weather_city_clean.info()

weather_city shape: (52584, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52584 entries, 0 to 52583
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp_local      52584 non-null  datetime64[ns]
 1   Temp (°C)            48840 non-null  float64       
 2   Dew Point Temp (°C)  48840 non-null  float64       
 3   Rel Hum (%)          48840 non-null  float64       
 4   Precip. Amount (mm)  48794 non-null  float64       
 5   Wind Dir (10s deg)   0 non-null      float64       
 6   Wind Spd (km/h)      0 non-null      float64       
 7   Visibility (km)      0 non-null      float64       
 8   Stn Press (kPa)      48840 non-null  float64       
dtypes: datetime64[ns](1), float64(8)
memory usage: 3.6 MB


In [12]:
print("weather_intl_a shape:", weather_intl_clean.shape)
weather_intl_clean.info()

weather_intl_a shape: (52584, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52584 entries, 0 to 52583
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp_local      52584 non-null  datetime64[ns]
 1   Temp (°C)            48792 non-null  float64       
 2   Dew Point Temp (°C)  48792 non-null  float64       
 3   Rel Hum (%)          48792 non-null  float64       
 4   Precip. Amount (mm)  0 non-null      float64       
 5   Wind Dir (10s deg)   47945 non-null  float64       
 6   Wind Spd (km/h)      48786 non-null  float64       
 7   Visibility (km)      48785 non-null  float64       
 8   Stn Press (kPa)      48792 non-null  float64       
dtypes: datetime64[ns](1), float64(8)
memory usage: 3.6 MB


In [13]:
# Null % across the full file (2021-2026) - the Temp/Dew Point/Rel Hum/Stn Press gap looks
# surprisingly large here, worth a closer look before deciding what to do about it
pd.DataFrame({
    "weather_city_pct_null": (weather_city_clean.isna().mean() * 100).round(1),
    "weather_intl_a_pct_null": (weather_intl_clean.isna().mean() * 100).round(1),
})

,weather_city_pct_null,weather_intl_a_pct_null
timestamp_local,0.0,0.0
Temp (°C),7.1,7.2
Dew Point Temp (°C),7.1,7.2
Rel Hum (%),7.1,7.2
Precip. Amount (mm),7.2,100.0
Wind Dir (10s deg),100.0,8.8
Wind Spd (km/h),100.0,7.2
Visibility (km),100.0,7.2
Stn Press (kPa),7.1,7.2


The 100% gaps above are expected - each is a variable the other station simply doesn't measure, and there's nothing to interpolate from. The Temp/Dew Point/Rel Hum/Stn Press gap looks like ~7% here, but that's misleading: this file runs through the end of 2026, and there's one long unbroken gap covering 2026-07 to 2026-12 (data that just hasn't been recorded yet for the future). Restricting to the real 2021-2025 modeling window, the actual gap is tiny - 0.09% in city (6 hours at most in a row) and 0.01% in intl_a (single isolated hours).

Because these are short, local gaps, filling them with a same-column time interpolation only needs the immediate neighboring hours - it doesn't rely on any statistic computed from the full dataset, so there's no train/test leakage risk in doing it now rather than after the fold split. `limit_area="inside"` keeps this safe: it only fills gaps with a real value on both sides, so the trailing 2026 block (no value after it) is left untouched rather than incorrectly extrapolated.

In [14]:
weather_city_clean = clean_weather(weather_city_raw)
weather_intl_clean = clean_weather(weather_intl_raw)

# Re-check null % restricted to the real 2021-2025 window, where the interpolation actually matters
end_2025 = "2025-12-31 23:00:00"
pd.DataFrame({
    "weather_city_pct_null": (weather_city_clean[weather_city_clean["timestamp_local"] <= end_2025].isna().mean() * 100).round(2),
    "weather_intl_a_pct_null": (weather_intl_clean[weather_intl_clean["timestamp_local"] <= end_2025].isna().mean() * 100).round(2),
})

,weather_city_pct_null,weather_intl_a_pct_null
timestamp_local,0.00,0.00
Temp (°C),0.00,0.00
Dew Point Temp (°C),0.00,0.00
Rel Hum (%),0.00,0.00
Precip. Amount (mm),0.09,100.00
Wind Dir (10s deg),100.00,1.67
Wind Spd (km/h),100.00,0.01
Visibility (km),100.00,0.02
Stn Press (kPa),0.00,0.00


## 3. Consumption

Six FSA files this time. `load_consumption()` already collapses each one to a single row per FSA-hour (summing `TOTAL_CONSUMPTION` and `PREMISE_COUNT` across `CUSTOMER_TYPE`/`PRICE_PLAN`), so what shows up here is that already-consolidated version, not the raw file - starting point for cleaning is a bit further along than it was for calendar and weather.

In [15]:
consumption_m5s_raw = load_consumption("M5S")

print("shape:", consumption_m5s_raw.shape)
consumption_m5s_raw.info()

shape: (43824, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   FSA                43824 non-null  object 
 1   timestamp_local    43824 non-null  object 
 2   TOTAL_CONSUMPTION  43824 non-null  float64
 3   PREMISE_COUNT      43824 non-null  int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 1.3+ MB


In [16]:
consumption_m5s_raw.head()

,FSA,timestamp_local,TOTAL_CONSUMPTION,PREMISE_COUNT
0,M5S,2021-01-01 00:00:00,4186.2,5927
1,M5S,2021-01-01 01:00:00,3868.8,5927
2,M5S,2021-01-01 02:00:00,3652.6,5927
3,M5S,2021-01-01 03:00:00,3538.7,5927
4,M5S,2021-01-01 04:00:00,3417.8,5927


Checking the same thing across all six FSAs before deciding anything, the same way calendar and weather were compared.

In [17]:
FSAS = ["M5S", "M5R", "M6G", "L4T", "M9W", "M9R"]
consumption_raw = {fsa: load_consumption(fsa) for fsa in FSAS}

pd.DataFrame([
    {
        "FSA": fsa,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "TOTAL_CONSUMPTION_nulls": df["TOTAL_CONSUMPTION"].isna().sum(),
        "PREMISE_COUNT_nulls": df["PREMISE_COUNT"].isna().sum(),
        "timestamp_local_dtype": df["timestamp_local"].dtype,
    }
    for fsa, df in consumption_raw.items()
])

,FSA,rows,cols,TOTAL_CONSUMPTION_nulls,PREMISE_COUNT_nulls,timestamp_local_dtype
0,M5S,43824,4,0,0,object
1,M5R,43824,4,0,0,object
2,M6G,43824,4,0,0,object
3,L4T,43824,4,0,0,object
4,M9W,43824,4,0,0,object
5,M9R,43824,4,0,0,object


Nothing to drop and nothing to fill here - the only fix needed is `timestamp_local`, still plain text like it was in calendar and weather. Outlier review (zero/negative consumption, implausible spikes) is a distribution question, so that's left for EDA rather than this cleaning pass.

In [18]:
consumption_clean = {fsa: clean_consumption(df) for fsa, df in consumption_raw.items()}

pd.DataFrame([
    {"FSA": fsa, "shape": df.shape, "timestamp_local_dtype": df["timestamp_local"].dtype}
    for fsa, df in consumption_clean.items()
])

,FSA,shape,timestamp_local_dtype
0,M5S,"(43824, 4)",datetime64[ns]
1,M5R,"(43824, 4)",datetime64[ns]
2,M6G,"(43824, 4)",datetime64[ns]
3,L4T,"(43824, 4)",datetime64[ns]
4,M9W,"(43824, 4)",datetime64[ns]
5,M9R,"(43824, 4)",datetime64[ns]


## 4. Joining the three cleaned datasets

With calendar, weather, and consumption each cleaned on their own, the last step is putting them together. For each FSA, consumption is joined with calendar and with the correct weather station (city or intl_a, per `FSA_TO_STATION`) on `timestamp_local`. Consumption drives the join rather than the other way around, since it's the dataset that defines the real 2021-2025 window - calendar and weather's extra 2026 rows never have a matching consumption row, so they simply don't appear in the result.

In [19]:
m5s_joined = build_fsa_hourly_dataset("M5S")

print("shape:", m5s_joined.shape)
m5s_joined.info()

shape: (43824, 47)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 47 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   FSA                      43824 non-null  object        
 1   timestamp_local          43824 non-null  datetime64[ns]
 2   TOTAL_CONSUMPTION        43824 non-null  float64       
 3   PREMISE_COUNT            43824 non-null  int64         
 4   year                     43824 non-null  int64         
 5   quarter                  43824 non-null  int64         
 6   month                    43824 non-null  int64         
 7   week_of_year             43824 non-null  int64         
 8   day_of_year              43824 non-null  int64         
 9   day_of_month             43824 non-null  int64         
 10  hour                     43824 non-null  int64         
 11  weekday                  43824 non-null  int64         
 12  is_weekend   

In [20]:
m5s_joined.head()

,FSA,timestamp_local,TOTAL_CONSUMPTION,PREMISE_COUNT,year,quarter,month,week_of_year,day_of_year,day_of_month,...,month_sin,month_cos,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Precip. Amount (mm),Wind Dir (10s deg),Wind Spd (km/h),Visibility (km),Stn Press (kPa)
0,M5S,2021-01-01 00:00:00,4186.2,5927,2021,1,1,53,1,1,...,0.0,1.0,-0.9,-5.8,69.0,0.0,NaN,NaN,NaN,101.67
1,M5S,2021-01-01 01:00:00,3868.8,5927,2021,1,1,53,1,1,...,0.0,1.0,-0.8,-5.3,71.0,0.0,NaN,NaN,NaN,101.66
2,M5S,2021-01-01 02:00:00,3652.6,5927,2021,1,1,53,1,1,...,0.0,1.0,-0.9,-5.9,69.0,0.0,NaN,NaN,NaN,101.69
3,M5S,2021-01-01 03:00:00,3538.7,5927,2021,1,1,53,1,1,...,0.0,1.0,-0.8,-4.9,74.0,0.0,NaN,NaN,NaN,101.77
4,M5S,2021-01-01 04:00:00,3417.8,5927,2021,1,1,53,1,1,...,0.0,1.0,-0.3,-4.3,75.0,0.0,NaN,NaN,NaN,101.75


Pooling all six FSAs together into the single dataset the rest of the project builds on.

In [21]:
fsa_hourly_master = build_all_fsa_dataset()

print("shape:", fsa_hourly_master.shape)
fsa_hourly_master["FSA"].value_counts()

shape: (262944, 47)


FSA
M5S    43824
M5R    43824
M6G    43824
L4T    43824
M9W    43824
M9R    43824
Name: count, dtype: int64

Every column should be null-free except the four weather measurements that are only ever reported by one of the two stations - checking that's exactly what's left.

In [22]:
null_report = fsa_hourly_master.isna().sum()
null_report = null_report[null_report > 0].sort_values(ascending=False)
pd.DataFrame({"nulls": null_report, "pct": (null_report / len(fsa_hourly_master) * 100).round(1)})

,nulls,pct
Wind Dir (10s deg),133668,50.8
Precip. Amount (mm),131595,50.0
Visibility (km),131493,50.0
Wind Spd (km/h),131490,50.0


Saving the final dataset to `data/interim/` (gitignored) so the rest of the project can load it directly instead of rebuilding it from the raw files every time.

In [23]:
output_path = Path.cwd().parents[1] / "data" / "interim" / "fsa_hourly_master.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)
fsa_hourly_master.to_parquet(output_path, index=False)

print("saved to:", output_path)
print("shape:", fsa_hourly_master.shape)

saved to: C:\Tessy\Master of Data Analytics\TERM V\ontario-electricity-peak-risk\data\interim\fsa_hourly_master.parquet
shape: (262944, 47)
